# COMPUTACIÓN CUÁNTICA - OPTATIVA - GRADOS ING. INFORMÁTICA Y SOFTWARE

# _Notebook_ sobre vectores de estado y esfera de Bloch

## Entorno de trabajo

In [ ]:
# No olvidar que en "google colab" hay que instalar qiskit
# Instala la última versión disponible en collab (2.1)
try:
    import google.colab
    print("In colab, let's install things...")
    #
    !pip install qiskit==2.1.2 qiskit[visualization] qiskit-aer qiskit-ibm-runtime qiskit-optimization matplotlib numpy scipy sympy pandas
except ImportError:
    print("NOT in colab")

In [ ]:
# Print package versions
import importlib
packages = [
    "qiskit",
    "qiskit_aer",
    "qiskit_ibm_runtime",
    "qiskit_optimization",
]
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:25s} : {mod.__version__}")
    except ImportError:
        print("%-25s: %s"%(pkg, "not installed"))


## Paquetería

In [ ]:
import numpy as np
import scipy.linalg as la
from IPython.display import display, Markdown, Latex, HTML
import matplotlib.pyplot as plt
import copy

from qiskit.visualization import array_to_latex, plot_bloch_multivector, plot_state_qsphere
from qiskit.quantum_info import Statevector, schmidt_decomposition

In [ ]:
# Alguna configuración extra

# Esto es para que al imprimir un objeto matplotlib se redondee y ocultar errores numéricos
np.set_printoptions(suppress=True, precision=5)

# Vectores de estado y la esfera de Bloch

Para visualizar vectores en la esfera de Bloch tenemos varias opciones. Una de ellas es un [visualizador online en Java:](https://javafxpert.github.io/grok-bloch/?fbclid=IwAR1lWfCug1lfTXxXenaJxnsdJh2g3he6lBJmjnjL_Jus0gOHR9u3UuaT7qs).

Aquí vamos a centrarnos en la clase <a href=https://qiskit.org/documentation/stubs/qiskit.quantum_info.Statevector.html>statevector</a> de *qiskit* que cuenta con un *visualizador* de la esfera de Bloch. La clase  acepta una lista de dos números complejos como componentes de un vector.  


## La clase Statevactor de Qiskit

In [ ]:
# Ejemplo de vector de estado |+>
state_plus =[1/np.sqrt(2),1/np.sqrt(2)]

statevector_plus = Statevector(state_plus)

In [ ]:
statevector_plus.draw('latex')

In [ ]:
statevector_plus.draw('latex_source')

In [ ]:
# Para escribirlo en terminal sin ser en un Jupiter
print(statevector_plus.draw('text'))

In [ ]:
statevector_plus.draw('Bloch')

El método `statevector_plus.draw('Bloch')` nos devuelve una figura de Matplotlib, así que podemos usar métodos como `savefig()`

In [ ]:
!pwd
statevector_plus.draw('Bloch').savefig("/tmp/Fig_Bloch_plus_state")
!ls -l /tmp/Fig*

<a id='sec_Notebooks_Vectores_y_Bloch_2'></a>
## Normalización del vector

Esta rutina no verifica si el vector está normalizado, como podemos ver en el siguiente ejemplo.

In [ ]:
state_2 =[-1.45/np.sqrt(2),2/np.sqrt(2)]

statevector_2 = Statevector(state_2)
statevector_2.draw('latex')

In [ ]:
b = statevector_2.draw('bloch')
display(b)

In [ ]:
# Podemos comprobar si el vector está o no normalizado:
if statevector_2.is_valid():
    print("Este vector YA está normalizado")
else:
    print("Este vector NO está normalizado")

# Para normalizarlo dividimos por su norma
statevector_2_norm = statevector_2 / la.norm(statevector_2)

if statevector_2_norm.is_valid():
    print("El nuevo vector YA está normalizado")
else:
    print("El nuevo vector NO está normalizado")

# Vamos a comprobarlo visualmente
bloch = statevector_2_norm.draw('bloch')
bloch.set_size_inches(2.5, 2.5, forward=True) # Por si queremos escalar la esfers
display(bloch)

In [ ]:
# Observa que podemos acceder a los datos "crudos" del vector
display(statevector_2_norm.data)

# Observa que el objeto "raw" es un vector numpy
# esto es dimensionalidad=1
#
# Si quisiéramos tener el vector en formato ket hay
# que redimensionar a dimensionalidad=2, shape=nx1
#
display(statevector_2_norm.data.reshape(2,1))

## Estado a partir de los ángulos

Una forma de construir un estado normalizado es usando la parametrización

$$
|u\rangle = \cos{\frac{\theta}{2}}\, |0\rangle + e^{i\phi}\sin{\frac{\theta}{2}}\,|1\rangle
$$

Hagamos una función que nos de el estado a partir de los ángulos $\theta$ y $\phi$

In [ ]:
def state_from_angles(theta, phi):
    u0 = np.cos(theta/2)
    u1 = (np.exp(1j*phi))*np.sin(theta/2)

    uket = [u0,u1]
    return Statevector(uket)
statevector_y_plus = state_from_angles(np.pi/2, np.pi/2)
statevector_y_plus.draw('Latex')

In [ ]:
statevector_y_plus.draw('Bloch')

<div style="border:1pt black solid; padding:3pt;">
<b>Ejercicio:</b>
    
Consulta la documentación de la clase <span style="font-family:courier,monospace;">Statevector</span>:
[https://quantum.cloud.ibm.com/docs/es/api/qiskit/qiskit.quantum_info.Statevector](https://quantum.cloud.ibm.com/docs/es/api/qiskit/qiskit.quantum_info.Statevector)

Explora otras maneras de generar el vector de estado:    
- A partir de un entero indicando en nº de vector de la base computacional $\{|0\rangle, |1\rangle, |2\rangle ... \}$
- A partir de una etiqueta como '0','1' (base computacional),'+','-', (base X), 'r','l' (base Y)

Juega en estos casos con el parámetro de dimensión
</div>

In [ ]:
###### Escribe aquí la solución
# --- Single Qubit (dimensión 2) ---
sv_0 = Statevector.from_int(0, dims=2) # Estado |0>
sv_1 = Statevector.from_int(1, dims=2) # Estado |1>

print("Single Qubit - Estado |0>:")
display(sv_0.draw('latex'))

# --- Multi-Qubit (2 qubits -> dimensión 4) ---
# Genera el estado |3> = |11> en un espacio de dimensión 4
sv_2 = Statevector.from_int(1, dims=4)
sv_3 = Statevector.from_int(3, dims=4)

print("2 Qubits - Estado |2>:")
display(sv_2.draw('latex'))

print("2 Qubits - Estado |3> = |11>:")
display(sv_3.draw('latex'))

sv_mas = Statevector.from_label("r")
display(sv_mas.draw('latex'))



## Producto tensorial (de Kronecker)

In [ ]:
# Definimos una función para visualizar el estado y la esfera a la vez
# Permite escalar la(s) esferas(s) con 'inches' pulgadas, si este parámetro
# no se pone, se dibuja al tamaño por defecto
# Si se pone 0, no se dibuja la esfera
def my_display_state_vector(psi, inches=None):
    d_latex = psi.draw('latex', prefix=r"|\psi\rangle = ")
    display(d_latex)
    #
    d_bloch = psi.draw('bloch')
    if isinstance(inches, (int,float)):
        if inches > 0:
            d_bloch.set_size_inches(inches, inches, forward=True)
            display(d_bloch)
    else:
        display(d_bloch)

In [ ]:
# Generamos dos vectores de estado
v1 = np.array([-1, 0.5])
v2 = np.array([1, 1-1j])
s1 = Statevector(v1/la.norm(v1))
s2 = Statevector(v2/la.norm(v2))

my_display_state_vector(s1, 2)
my_display_state_vector(s2, 2)

In [ ]:
# Vamos a ver diferentes formas de realizar el producto de kronecker

# La primera, usando el operador circunflejo ^
sc1 = s1 ^ s2
sc2 = s1 ^ s2 ^ s2

# Dibujar las multiesferas
my_display_state_vector(sc1,3)
my_display_state_vector(sc2,4.25)

<div style="border:1pt black solid; padding:3pt;">
    
<b>Ejercicio:</b>
    
Observa las esferas de Bloch y el producto realizado:
- ¿en qué orden de numeración están los qubits en el estado computesto?
- ¿cuál es el más significativo (MSB) y el menos significativo (LSB)?

</div>

### Orden de numeración de los qubits en el estado compuesto

Al realizar el producto tensorial con el operador `^` de Qiskit (`sc1 = s1 ^ s2`), los qubits se asignan **de derecha a izquierda** en la ordenación de índices:

* **`qubit 0`** corresponde al estado **`s2`** (el último operando de la expresión `s1 ^ s2`). Como se aprecia en la figura, la esfera del `qubit 0` coincide con la representación individual de `s2`.


* **`qubit 1`** corresponde al estado **`s1`** (el primer operando de la expresión).


* En el caso de tres qubits (`sc2 = s1 ^ s2 ^ s2`), **`qubit 0`** es `s2`, **`qubit 1`** es `s2`, y **`qubit 2`** es `s1`.



---

### Identificación de LSB y MSB

Qiskit utiliza el convenio **Little-Endian** para escribir los estados dentro de la notación en ket $\vert{}q_{n-1}\dots q_1 q_0\rangle$:

* **LSB (*Least Significant Bit* / Qubit menos significativo):** Es el **`qubit 0`**, situado en la posición más a la derecha dentro de la etiqueta del ket ($q_0$). En los ejercicios mostrados corresponde al estado **`s2`**.


* **MSB (*Most Significant Bit* / Qubit más significativo):** Es el qubit con el subíndice más alto ($q_{n-1}$), situado en el extremo izquierdo de la etiqueta del ket:


* En `sc1` ($2$ qubits): El MSB es el **`qubit 1`** (estado **`s1`**).
* En `sc2` ($3$ qubits): El MSB es el **`qubit 2`** (estado **`s1`**).

In [ ]:
# Usando el método tensor()
#  self.tensor(other) = kron(self, other)

sc1_t = s1.tensor(s2)
sc2_t = s1.tensor(s2.tensor(s2))

my_display_state_vector(sc1_t, 0)
my_display_state_vector(sc2_t, 0)


In [ ]:
# Usando el método expand()
#   self.expand(other) = kron(other,self)

sc1_e = s2.expand(s1)
sc2_e = (s2.expand(s2)).expand(s1)

my_display_state_vector(sc1_e, 0)
my_display_state_vector(sc2_e, 0)


In [ ]:
# Comprobemos que todas las formas de calcular el producto
# tensorial dan el mismo resultado

assert(sc1 == sc1_e)
assert(sc2 == sc2_e)
assert(sc1 == sc1_t)
assert(sc2 == sc2_t)

<div style="border:1pt black solid; padding:3pt;">

<b>Ejercicio:</b>
    
La **descomposición de Schmidt** nos proporciona un criterio para saber si un estado perteneciente a un espacio bipartito $H=H_A \otimes H_B$ es **factorizable** o **entrelazado**.
(Ver [descomposición de Schmidt](https://quantum.cloud.ibm.com/learning/es/courses/general-formulation-of-quantum-information/purifications-and-fidelity/purifications#descomposiciones-de-schmidt)).

**¡Atención!** No confundir esta descomposición con el método de ortogonalizadión de Gram-Schmidt

En Qiskit se puede realizar dicha descomposición con este método:
[https://quantum.cloud.ibm.com/docs/es/api/qiskit/quantum_info#schmidt_decomposition](https://quantum.cloud.ibm.com/docs/es/api/qiskit/quantum_info#schmidt_decomposition)

👉 Lee la documentación de referencia de este método de descomposición, y busca información de cómo el resultado de dicha descomposición informa de si un estado está entrelazado o, por contra, es factorizable.

Considera que $H$ es de dimensión 4 (2 qubits), producto tensorial de dos espacios de dimensión 2 (1 qubit cada uno):

- Genera varios vectores de $H$ con el producto tensorial y comprueba que la descomposición de Schmidt nos indica que son factorizables.

- Construye también otros vectores de $H$ que no sean factorizables (un truco puede ser eliminando o añadiendo componentes de uno vectorizable, aunque no es condición suficiente). Comprueba que la factorización de Schmidt así lo indica.

<b> <font size="+1"><span style="color:red">[OPCIONAL, EJERCICIO DE NOTA] </span></font></b>

Si se tiene que el espacio de trabalo, $H_n$, es de dimensión $2^n$ (n qubits) producto tensorial de $n$ espacios de dimensión 2 (1 qubit cada uno) $H_n = H_1^{\otimes n}$ con $H_1$ de dimension 2:

- Razona, ¿cómo podríamos usar, en este caso, la descomposición de Schmidt para saber si un estado de $n$ qubits es entrelazado?

- Explica cómo lo harías y qué complejidad computacional implica. No hace falta desarrollar el código, sólo explicar qué habría que hacer.

</div>

In [ ]:
###### Escribe aquí la solución
# ==============================================================================
# 1. VECTORES FACTORIZABLES EN H = C² ⊗ C² (2 qubits)
# ==============================================================================

# Generación de vectores factorizables usando producto tensorial y etiquetas
v_sep1 = Statevector.from_label('0+')  # |0> ⊗ |+>
v_sep2 = Statevector.from_label('r1')  # |r> ⊗ |1>

# Función para verificar factorizabilidad con schmidt_decomposition
def probar_schmidt(statevector, nombre):
    # Descomposición tomando el qubit 0 como subsistema A (qargs=[0])
    decomp = schmidt_decomposition(statevector, qargs=[0])
    rango = len(decomp)
    coeficientes = [np.round(item[0], 4) for item in decomp]

    print(f"--- Estado {nombre} ---")
    print(f"Coeficientes de Schmidt: {coeficientes}")
    print(f"Rango de Schmidt (r): {rango}")
    if rango == 1:
        print(" Verdict: FACTORIZABLE / SEPARABLE\n")
    else:
        print(" Verdict: ENTRELAZADO\n")

probar_schmidt(v_sep1, "|0+>")
probar_schmidt(v_sep2, "|r1>")


# ==============================================================================
# 2. VECTORES NO FACTORIZABLES (ENTRELAZADOS) EN H = C² ⊗ C²
# ==============================================================================

# Estado de Bell |Φ+> = (|00> + |11>) / √2
v_ent1 = (Statevector.from_label("00")+Statevector.from_label("11"))/np.sqrt(2)

# Vector modificado a partir de uno separable para romper factorizabilidad
# Partimos de (|00> + |01> + |10> + |11>)/2 y modificamos/eliminamos componentes
v_ent2 = (Statevector.from_label("00")+Statevector.from_label("11")+Statevector.from_label("01")+Statevector.from_label("10"))/2

probar_schmidt(v_ent1, "Estado de Bell |Φ+>")
probar_schmidt(v_ent2, "Vector Modificado {|00>: 0.6, |11>: 0.8}")

### Ejercicio de Nota: Descomposición de Schmidt en $n$ Qubits ($H_n = H_1^{\otimes n}$)

#### 1. Razonamiento Teórico

Para un sistema de $n$ qubits ($H_n$ de dimensión $2^n$), la descomposición de Schmidt está definida para particiones **bipartitas** ($A \otimes B$).

Un estado puro de $n$ qubits es **completamente factorizable** (totalmente separable) si y solo si se puede escribir como producto tensorial individual de cada uno de sus qubits:

$$\vert{}\psi\rangle = \vert{}\psi_0\rangle \otimes \vert{}\psi_1\rangle \otimes \dots \otimes \vert{}\psi_{n-1}\rangle$$

Para comprobar si el estado está libre de cualquier entrelazamiento sin necesidad de probar todas las combinaciones posibles de particiones, basta con verificar si **cada qubit individual $k$ está separable del resto de los $n-1$ qubits**.

Si para **cada qubit $k \in \{0, 1, \dots, n-1\}$**, la partición bipartita formada por $A = \{k\}$ y $B = \{0, \dots, n-1\} \setminus \{k\}$ tiene un rango de Schmidt $r_k = 1$, entonces el qubit $k$ no está entrelazado con ningún otro subsistema. Si esto se cumple para todos los $k$, el estado es **completamente factorizable**.

#### 2. Procedimiento Algorítmico

1. Dado el `Statevector` de $n$ qubits (vector de dimensión $2^n$).
2. Para cada índice de qubit $k$ de $0$ a $n-1$:
* Definir la partición bipartita $A = \{k\}$ (dimensión 2) y $B = \{0, \dots, n-1\} \setminus \{k\}$ (dimensión $2^{n-1}$).
* Reorganizar las $2^n$ amplitudes en una matriz $M_k$ de dimensión $2 \times 2^{n-1}$.
* Calcular los valores singulares de $M_k$ (o aplicar `schmidt_decomposition(state, qargs=[k])`).
* **Evaluación:** Si el número de valores singulares mayores que cero (el rango de Schmidt $r_k$) es mayor que $1$, **el estado está entrelazado** y el proceso finaliza inmediatamente.


3. Si para los $n$ qubits se obtiene $r_k = 1$, el estado es **completamente factorizable**.

#### 3. Complejidad Computacional

* **Dimensión de la matriz por bipartición:** Para el qubit $k$, la matriz $M_k$ tiene dimensión $2 \times 2^{n-1}$.
* **Descomposición en Valores Singulares (SVD):**
* Para una matriz de dimensión $2 \times 2^{n-1}$, calcular sus valores singulares mediante $M_k M_k^\dagger$ requiere multiplicar una matriz de $2 \times 2^{n-1}$ por su traspuesta conjugada de $2^{n-1} \times 2$, lo cual toma $\mathcal{O}(2^n)$ operaciones flotantes.
* La obtención de los autovalores de la matriz resultante de $2 \times 2$ es de orden $\mathcal{O}(1)$.


* **Complejidad Temporal Total:**
* Realizar esta verificación para los $n$ qubits individuales toma $n \times \mathcal{O}(2^n) = \mathbf{\mathcal{O}(n 2^n)}$ operaciones.


* **Complejidad Espacial:**
* Almacenar el vector de estado de $n$ qubits requiere $\mathbf{\mathcal{O}(2^n)}$ de memoria RAM.

## Representación estados no factorizables

Veamos cómo se pueden representar los estados no factorizables _(entangled)_ visualmente.

In [ ]:
e0 = Statevector([1, 0], dims=2)  # dims indica la dimensión del estado
e1 = Statevector([0, 1], dims=2)

ebell = ((e0^e0)+(e1^e1))/np.sqrt(2)
# Alternativa
# ebell = Statevector([1/np.sqrt(2), 0, 0, 1/np.sqrt(2)])

# Como hay entanglement, ¿se pueden representar los qubits por separado?
# Observa el resultado de la(s) esfera(s) de Bloch
my_display_state_vector(ebell,3)

In [ ]:
# En estos casos, una forma de visualizar el estado (en Qiskit)
# es usando una Q-esfera.
#
# Cada amplitud se representa por un circulo, cuyo diámetro es proporcional a
# la magnitud de la amplitud. La fase se representa mediante un código de colores.
display(plot_state_qsphere(ebell, figsize=(4,4)))

In [ ]:
# Mostremos la Q-sphere de un estado con aspecto extraño
emix = e0 + (1+1j)*e1
eleftminus = Statevector.from_label('l') + Statevector.from_label('-')
strangestate = emix^ebell^eleftminus
strangestate = strangestate/la.norm(strangestate)
display(strangestate.draw('latex'))
fig = plot_state_qsphere(strangestate, figsize=(4,4))
display(fig)

# Vamos a imprimirla desde otro ángulo
# - current axes
ax = fig.get_axes()[0]
# - cambiar azimut (angulo lateral) y elevación (ángulo arriba/abajo)
ax.view_init(elev=-55, azim=120)
# - dibujar de nuevo
display(fig)

<div style="border:1pt black solid; padding:3pt;">

<b>Ejercicio: El estado GHZ</b>

Un popular estado entrelazado de 3 qubits es el estado de __Greenberger–Horne–Zeilinger__ (GHZ) que viene definido como:

$$ |\text{GHZ}\rangle = \frac{|000\rangle - |111\rangle}{\sqrt{2}} $$

Se pide construir un `Statevector` en que represente esta variante GHZ de 3 cúbits y visualizar su Q-esfera mediante el método `plot_state_qsphere`.

Analiza la esfera resultante, fíjate en aspectos como:
  * ¿Cuántos puntos aparecen en la Q-esfera y por qué?
  * ¿Qué indican el tamaño y el color de los puntos sobre las amplitudes y fases del estado?

</div>

In [ ]:
###### Escribe aquí la solución
evGHZ = (Statevector.from_label("000")-Statevector.from_label("111"))/np.sqrt(2)
display(evGHZ.draw('latex'))
new_fig = plot_state_qsphere(evGHZ, figsize = (4,4))
display(new_fig)
# Vamos a imprimirla desde otro ángulo
# - current axes
ax = new_fig.get_axes()[0]
# - cambiar azimut (angulo lateral) y elevación (ángulo arriba/abajo)
ax.view_init(elev=-55, azim=120)
# - dibujar de nuevo
display(new_fig)

### Análisis de la Q-esfera resultante

1. **¿Cuántos puntos aparecen en la Q-esfera y por qué?**
* Aparecen exactamente 2 puntos.


* **Explicación:** Un registro de 3 qubits abarca un espacio de Hilbert de dimensión $2^3 = 8$ estados de la base computacional ($\vert{}000\rangle, \vert{}001\rangle, \dots, \vert{}111\rangle$). La Q-esfera representa únicamente los estados de la base cuya **amplitud de probabilidad es diferente de cero**. Dado que la superposición del estado $\vert{}\text{GHZ}\rangle$ contiene solo los términos $\vert{}000\rangle$ (en el polo norte) y $\vert{}111\rangle$ (en el polo sur), los otros 6 estados tienen amplitud 0 y no se dibujan.




2. **¿Qué indican el tamaño y el color de los puntos sobre las amplitudes y fases?**
* **Tamaño del punto:** Es proporcional al **módulo de la amplitud de probabilidad** ($\vert{}c_i\vert{}$) o a la probabilidad de medir dicho estado ($\vert{}c_i\vert{}^2$). En este estado, $\vert{}000\rangle$ y $\vert{}111\rangle$ poseen coeficientes de magnitud $\vert{}1/\sqrt{2}\vert{} = \vert{}-1/\sqrt{2}\vert{} \approx 0.707$ (lo que equivale a una probabilidad del $50\%$ cada uno), por lo que **ambos puntos tienen el mismo tamaño**.


* **Color del punto:** Corresponde a la **fase compleja** ($\phi$) de la amplitud del término $c_i = \vert{}c_i\vert{}e^{i\phi}$.


* El término $\vert{}000\rangle$ tiene una amplitud real positiva ($+1/\sqrt{2}$), correspondiente a una fase de $\phi = 0$ radianes.


* El término $\vert{}111\rangle$ tiene una amplitud real negativa ($-1/\sqrt{2} = \frac{1}{\sqrt{2}}e^{i\pi}$), correspondiente a un desfase de $\phi = \pi$ radianes ($180^\circ$). Por esta razón, el punto del polo sur aparece teñido con un color totalmente opuesto en la rueda cromática respecto al del polo norte.